# 2022_12_30

데이터 변경 
* 기상청 데이터 : 단기 예보 -> 단기 실황  , 국제 표준시 사용하는 시간대를 한국 시간대로 변경  


## spatial_agg_apply

1. 행정동에 대해서 인접한 행정동을 dict로 저장  
2. (시간대, 동) 인덱스로 접근을 하여 같은 시간대에 인접한 동을 가져옴 
3. 가져온 동 count값을 다 더함  
4. count 값에 가중치를 곱함  


#### sampliing
강남동과 강남동에 인접한 행정동에서 나타난 count 값을 시각화

![](https://i.imgur.com/Mcv8LkH.png)

이웃한 동에 대해서 가중치를 0.5로 해서 계산
![](https://i.imgur.com/B1rog8w.png)


하나의 데이터를 생성하는데 10시간 정도 걸림 -> 파일을 나눠서 처리함  


## training 

1. 새로 만든 데이터에가 spatial aggregation을 추가

다음과 같이 variable에 `nei1`(sptial aggeration) 추가
```python
def data_processing(path : str  , unit : float , ewma_fun : moving_average_alpha ):
    # path  : 데이터가 저장된 경로
    # unit  : ewma의 unit
    
    data = pd.read_csv(path)    
    # 사용하는 column만 선택
    data = data[['REG_DTIME', 'h_dong', 'count', 'pops', 'windspd','humid', 'temp', 'precip_form', 'precip', 'isHoliday' , 'nei1']]
    data['REG_DTIME'] = pd.to_datetime(data['REG_DTIME'])
    data['DOW'] = data['REG_DTIME'].dt.dayofweek
    data['HOD'] = data['REG_DTIME'].dt.hour
    data['MOY'] = data['REG_DTIME'].dt.month
    # time_idx 시간을 고유하게 표현(윤년)
    #for x , y in enumerate(data['REG_DTIME'].unique()):
    #    t_data_index = data[data['REG_DTIME'] == y].index
    #    data.loc[t_data_index , 'time_idx'] = x
    data["time_idx"] =  \
    data["REG_DTIME"].dt.year * 365*24 + \
    data["REG_DTIME"].dt.day_of_year * 24  + \
    data["REG_DTIME"].dt.hour 
    data["time_idx"] -= data["time_idx"].min()
    # trainer에 넣기 위해서 category로 만들기
    data['h_dong'] = data['h_dong'].astype(str)
    data['DOW'] = data['DOW'].astype(str)
    data['HOD'] = data['HOD'].astype(str)
    data['MOY'] = data['MOY'].astype(str)
    data['precip_form'] = data['precip_form'].astype(str)
    data['isHoliday'] = data['isHoliday'].astype(str)
    # ewma 적용
    if unit == 0:
        return data
    else:
        data = ewma_fun(data,unit)
        return data    
    

def get_training(data , max_prediction_length = 24 , max_encoder_length = 24*7):
    # traing data 생성
    training_cutoff = data["time_idx"].max() - max_prediction_length
    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = "count",
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=24,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=["HOD", "DOW" , 'isHoliday', 'MOY'],
        time_varying_known_reals=['pops'],
        time_varying_unknown_categoricals=['precip_form'],
        time_varying_unknown_reals=['count','windspd' , 'temp' ,'precip', 'nei1'], 
        target_normalizer=GroupNormalizer(
            groups=["h_dong"], 
            transformation="relu",
            center = False
        ),
        add_relative_time_idx=True,
        add_target_scales=False,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True,
        #predict_mode = False
        )
    return training

```

2. 학습 

> 기존의 result : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_15_TEST_ewma_alpha/result_veiw.ipynb

> result : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_28_TEST_new_data_training_spatial_aggregate/result_veiw.ipynb
